# Milestone 2 — Transformers, Pretrained Embeddings & Zero-Shot Classification

**Smart MCQ Solver Challenge** | Roll no: `22f3000114`

**Task:** For each question (a `prompt` + 5 options A–E), predict the top-3 most likely answers in ranked order. Metric: **MAP@3**.

---

## What this milestone demonstrates (per M2 spec)
1. **HuggingFace `transformers` + `datasets`** libraries.
2. **BERT/RoBERTa architecture + attention** (discussed in analysis).
3. **Pretrained context-aware embeddings** to score prompt↔option similarity — *Approach A*.
4. **Zero-shot classification** with an NLI model — *Approach B*.

## The story so far (from Milestone 1)
M1 established baselines on the **same train set**:

| Method | MAP@3 |
|--------|-------|
| Majority class (`B C A`) | **0.4213** |
| Word2Vec + cosine | 0.3154 |
| TF-IDF + cosine | 0.2910 |

**Key M1 finding:** similarity-to-prompt baselines *lose* to a dumb majority-class guess, because distractors are engineered to share vocabulary with the prompt. "Most similar to prompt" is a misleading signal. M2 tests whether **context-aware transformer embeddings** and **an NLI model that reasons about the prompt↔option relationship** can beat that 0.4213 wall.

We reuse the **exact same** `clean_text`, `map_at_3`, data loading, and W&B scaffolding from M1 so all runs land in the same project and compare directly (feeds L1 checkpoint #2: ≥3 compared runs).

## 0. Setup & imports

In [ ]:
!pip install -qU sentence-transformers transformers datasets wandb

In [ ]:
import os, re, glob, numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, accuracy_score

DEVICE = 0 if torch.cuda.is_available() else -1
print("CUDA available:", torch.cuda.is_available(), "| device:", DEVICE)

OPTIONS = ["A", "B", "C", "D", "E"]

## 1. Load data (robust Kaggle path handling — reused from M1)

In [ ]:
def find_train():
    for pat in ['/kaggle/input/**/train.csv', './train.csv', 'train.csv']:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError('train.csv not found')

path = find_train()
print('Loading:', path)
df = pd.read_csv(path)
print('Shape:', df.shape)
df.head(2)

In [ ]:
# Missing-data / integrity check (reused from M1)
print('Nulls per column:\n', df.isnull().sum())
print('\nAnswer label distribution:\n', df['answer'].value_counts())
assert df['answer'].isin(OPTIONS).all(), 'Unexpected answer label'
true_labels = df['answer'].tolist()

## 2. Shared scaffolding (reused verbatim from M1)

`clean_text` and `map_at_3` are identical to M1 so comparisons are apples-to-apples. `map_at_3` carries the same assertion tests.

In [ ]:
def clean_text(text):
    """Lowercase, strip the boilerplate MCQ wrapper, collapse whitespace.
    Identical to the M1 cleaner for consistency."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'pick the best possible answer:\s*', '', text)
    text = re.sub(r'\s*among the listed options\.?', '', text)
    text = re.sub(r'[^a-z0-9\s.,;:%()\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
def map_at_3(predictions, labels):
    """Mean Average Precision @ 3.
    predictions: list of ranked label lists e.g. ['B','C','A'].
    labels: list of true labels e.g. 'B'.
    Rank1=1.0, Rank2=0.5, Rank3=1/3, else 0."""
    weights = [1.0, 1/2, 1/3]
    total = 0.0
    for preds, true in zip(predictions, labels):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = weights[i]
                break
        total += score
    return total / len(labels)

# --- assertion tests (from M1) ---
assert abs(map_at_3([['A','B','C']], ['A']) - 1.0) < 1e-9
assert abs(map_at_3([['A','B','C']], ['B']) - 0.5) < 1e-9
assert abs(map_at_3([['A','B','C']], ['C']) - 1/3) < 1e-9
assert abs(map_at_3([['A','B','C']], ['D']) - 0.0) < 1e-9
print('map_at_3 assertions passed.')

In [ ]:
def scores_to_ranked(score_matrix):
    """score_matrix: (n_questions, 5) higher = more likely.
    Returns list of ranked-label lists (top 3 first, but full order kept)."""
    ranked = []
    order = np.argsort(-score_matrix, axis=1)  # descending
    for row in order:
        ranked.append([OPTIONS[i] for i in row])
    return ranked

def evaluate(score_matrix, name):
    """Compute MAP@3, top-1 accuracy, macro-F1 from a score matrix."""
    ranked = scores_to_ranked(score_matrix)
    top1 = [r[0] for r in ranked]
    m = map_at_3(ranked, true_labels)
    acc = accuracy_score(true_labels, top1)
    f1 = f1_score(true_labels, top1, labels=OPTIONS, average='macro')
    print(f'{name:35s} | MAP@3={m:.4f} | acc={acc:.4f} | macroF1={f1:.4f}')
    return {'map_at_3': m, 'accuracy': acc, 'macro_f1': f1, 'ranked': ranked}

## 3. Weights & Biases setup (reused from M1)

Same entity/project as M1 so M2 runs sit alongside M1's three baselines.

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret('WANDB')
    wandb.login(key=wandb_key)
    WANDB_OK = True
except Exception as e:
    print('W&B secret not found — running without logging. (', e, ')')
    WANDB_OK = False

WANDB_ENTITY = '22f3000114-dl-genai-project'
WANDB_PROJECT = 'dl-genai-project'

def log_run(run_name, config, metrics):
    if not WANDB_OK:
        print('[skip W&B]', run_name, metrics)
        return
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=run_name, config=config, reinit=True)
    run.log({k: v for k, v in metrics.items() if k != 'ranked'})
    run.finish()

## 4. Approach A — Pretrained sentence embeddings + cosine similarity

The **direct upgrade from M1**: same ranking logic (cosine of prompt vs each option), but TF-IDF/Word2Vec vectors are replaced by **context-aware embeddings** from `all-MiniLM-L6-v2` (the bootcamp Day 3/4 model). This isolates one variable — *does a better embedding beat the M1 baselines?*

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',
                               device='cuda' if DEVICE == 0 else 'cpu')

# Clean prompt + all 5 options
prompts_clean = [clean_text(p) for p in df['prompt']]
options_clean = {opt: [clean_text(t) for t in df[opt]] for opt in OPTIONS}

In [ ]:
# Embed prompts once, and each option column once (batched, normalized)
prompt_emb = st_model.encode(prompts_clean, batch_size=64,
                             convert_to_numpy=True, normalize_embeddings=True,
                             show_progress_bar=True)

option_emb = {}
for opt in OPTIONS:
    option_emb[opt] = st_model.encode(options_clean[opt], batch_size=64,
                                      convert_to_numpy=True,
                                      normalize_embeddings=True,
                                      show_progress_bar=False)

# Cosine sim = dot product (already normalized). Build (n, 5) score matrix.
scoresA = np.zeros((len(df), 5))
for j, opt in enumerate(OPTIONS):
    scoresA[:, j] = np.sum(prompt_emb * option_emb[opt], axis=1)

resA = evaluate(scoresA, 'A: MiniLM prompt-option cosine')

In [ ]:
log_run('m2-A-minilm-cosine',
        config={'milestone': 2, 'approach': 'A-pretrained-embeddings',
                'model': 'all-MiniLM-L6-v2', 'ranking': 'prompt-option-cosine'},
        metrics=resA)

## 5. Approach B — Zero-shot NLI scoring

The **"understanding" upgrade**. Instead of measuring surface similarity, we ask an NLI model whether each option *entails* being the correct answer.

**Framing (explained for viva):** each option here is a full declarative factual statement, and the distractors differ from the true answer on subtle facts. So we set:
- **premise** = the option text (a factual claim)
- **hypothesis** = `"This correctly answers the question: {prompt}"`

and score the **entailment** probability. The option the model most believes correctly answers the prompt should win. We use `facebook/bart-large-mnli` via the zero-shot pipeline, scoring all 5 options per question on the full 2000-row train set (batched).

In [ ]:
from transformers import pipeline

zshot = pipeline('zero-shot-classification',
                 model='facebook/bart-large-mnli', device=DEVICE)

# Build (premise=option, hypothesis about prompt) pairs.
# We run each option as a single-candidate zero-shot call and read the
# entailment score. To batch efficiently we pass the option text as the
# sequence and a prompt-derived hypothesis template.
def make_hypothesis(prompt):
    return f'This correctly answers the question: {prompt}'

hyp_templates = [make_hypothesis(clean_text(p)) for p in df['prompt']]

In [ ]:
# For each option column, score entailment of (option -> hypothesis).
# We use the pipeline's hypothesis_template with a fixed label so the
# returned score is the entailment probability for that option.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

nli_name = 'facebook/bart-large-mnli'
tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name)
nli = nli.to('cuda' if DEVICE == 0 else 'cpu').eval()

# bart-mnli label order: 0=contradiction, 1=neutral, 2=entailment
ENTAIL_IDX = 2

@torch.no_grad()
def entail_scores(premises, hypotheses, batch_size=32):
    """Return P(entailment) for each (premise, hypothesis) pair."""
    out = []
    for i in range(0, len(premises), batch_size):
        pb = premises[i:i+batch_size]
        hb = hypotheses[i:i+batch_size]
        enc = tok(pb, hb, return_tensors='pt', truncation=True,
                  max_length=256, padding=True).to(nli.device)
        logits = nli(**enc).logits
        probs = F.softmax(logits, dim=-1)[:, ENTAIL_IDX]
        out.extend(probs.cpu().numpy().tolist())
    return np.array(out)

In [ ]:
scoresB = np.zeros((len(df), 5))
for j, opt in enumerate(OPTIONS):
    premises = options_clean[opt]
    scoresB[:, j] = entail_scores(premises, hyp_templates, batch_size=32)
    print(f'Scored option {opt}')

resB = evaluate(scoresB, 'B: bart-large-mnli entailment')

In [ ]:
log_run('m2-B-nli-entailment',
        config={'milestone': 2, 'approach': 'B-zero-shot-NLI',
                'model': 'facebook/bart-large-mnli',
                'framing': 'premise=option, hypothesis=answers-prompt'},
        metrics=resB)

## 6. Results & analysis

Fill this table in after running on Kaggle. The M1 numbers are fixed baselines.

In [ ]:
summary = pd.DataFrame([
    {'milestone': 'M1', 'method': 'Majority class (B C A)', 'MAP@3': 0.4213, 'accuracy': None, 'macro_f1': None},
    {'milestone': 'M1', 'method': 'Word2Vec + cosine',      'MAP@3': 0.3154, 'accuracy': None, 'macro_f1': None},
    {'milestone': 'M1', 'method': 'TF-IDF + cosine',        'MAP@3': 0.2910, 'accuracy': None, 'macro_f1': None},
    {'milestone': 'M2', 'method': 'A: MiniLM cosine',       'MAP@3': resA['map_at_3'], 'accuracy': resA['accuracy'], 'macro_f1': resA['macro_f1']},
    {'milestone': 'M2', 'method': 'B: bart-mnli entailment','MAP@3': resB['map_at_3'], 'accuracy': resB['accuracy'], 'macro_f1': resB['macro_f1']},
])
summary.sort_values('MAP@3', ascending=False)

### Analysis (write up after seeing numbers)

**Attention / architecture note (for viva checkpoint #5):** BERT/RoBERTa and BART all use the Transformer self-attention mechanism — each token attends to every other token, producing *context-aware* representations (the vector for a word depends on its neighbours). This is exactly why M2's embeddings can, in principle, out-resolve M1's Word2Vec (which gives every occurrence of a word one static vector). MiniLM is a distilled BERT-family sentence encoder; bart-large-mnli is a seq2seq Transformer fine-tuned on the MultiNLI entailment task.

**Points to address in write-up:**
1. Did Approach A (MiniLM cosine) beat M1's Word2Vec (0.3154) and TF-IDF (0.2910)? If yes, context-aware embeddings help. Did it beat majority class (0.4213)?
2. Did Approach B (NLI) cross the 0.4213 majority-class wall? This is the real test of whether *reasoning about the prompt↔option relationship* beats *surface similarity*.
3. Why NLI might still struggle: the distractors are **near-paraphrases** differing on one buried fact ("10 kV" vs "100 kV", "fusion" vs "fission"). A general MNLI model isn't trained to detect factual correctness — it detects logical entailment of *form*, so several options can all look entailed. This limitation motivates M3 (RAG: retrieve the actual fact) and M4 (fine-tune on the task).
4. Accuracy/F1 vs MAP@3: note the gap — MAP@3 rewards having the answer anywhere in the top 3, so a model can have low top-1 accuracy but respectable MAP@3.

**Conclusion hook:** whatever the outcome, it sets up M3/M4. If pretrained models still can't beat majority-class, the finding is that *closed-book factual MCQ with adversarial distractors needs either retrieval (RAG) or task fine-tuning* — not just better embeddings.